# B3 — Words as vectors

**From:** "computers cannot read"  **To:** building word vectors from our own call transcripts and watching meaning emerge as geometry.

Models eat numbers. Words are symbols. The bridge between them is *the* founding move of modern NLP, and you can build it from scratch in this notebook with counting alone.

## Attempt 1: one-hot vectors (and why they fail)
Give each word its own slot: "hotel" = [1,0,0,…], "guesthouse" = [0,1,0,…]. Honest encoding, but every pair of words is equally distant — the geometry contains **no meaning**. We measure that with **cosine similarity**: 1.0 = same direction (similar), 0.0 = perpendicular (unrelated).

In [ ]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "rubric.yaml").exists())
import sys
sys.path.insert(0, str(ROOT / "pipeline"))
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(7)
print("ready · repo:", ROOT.name)

words = ["hotel", "guesthouse", "thursday", "friday", "cheap", "expensive"]
onehot = np.eye(len(words))
def cosine(u, v):
    return u @ v / (np.linalg.norm(u) * np.linalg.norm(v) + 1e-9)
print("cosine similarities under one-hot:")
print(f"  hotel ~ guesthouse: {cosine(onehot[0], onehot[1]):.2f}")
print(f"  hotel ~ thursday:   {cosine(onehot[0], onehot[2]):.2f}")
print("identical words aside, EVERYTHING is 0 - 'hotel' is no closer to 'guesthouse' than to 'thursday'")

## Attempt 2: "you shall know a word by the company it keeps" (Firth, 1957)
Words that appear in similar **contexts** have similar meanings — *thursday* and *friday* both follow "leave on", precede "at 9:30". So: slide a window over lots of text, count which words co-occur near which, and let each word's **row of counts** be its vector. Similar contexts → similar rows → high cosine. Meaning from raw counting.

Our corpus: 100 real SpokenWOZ call transcripts (loading the 246MB file takes a few seconds).

In [ ]:
import json, re
from collections import Counter
data = json.load(open(ROOT / "data" / "spokenwoz" / "data.json"))
dev_ids = open(ROOT / "data" / "spokenwoz" / "valListFile.json").read().split()[:100]
docs = []
for did in dev_ids:
    if did in data:
        text = " ".join(t["text"] for t in data[did]["log"])
        docs.append(re.findall(r"[a-z']+", text.lower()))
freq = Counter(w for d in docs for w in d)
ranked = [w for w, c in freq.most_common(230)]
vocab = [w for w in ranked[30:] if len(w) > 2][:150]   # drop the 30 most frequent: glue words ('the', 'and') co-occur with EVERYTHING and drown meaning
v2i = {w: i for i, w in enumerate(vocab)}
print(f"{len(docs)} calls · {sum(map(len, docs))} words · vocab of {len(vocab)} -> sample:", vocab[:12])

In [ ]:
WINDOW = 4
C = np.zeros((len(vocab), len(vocab)))
for doc in docs:
    idx = [v2i.get(w, -1) for w in doc]
    for i, wi in enumerate(idx):
        if wi < 0: continue
        for j in range(max(0, i - WINDOW), min(len(idx), i + WINDOW + 1)):
            if j != i and idx[j] >= 0:
                C[wi, idx[j]] += 1
V = np.log1p(C)                                   # tame the heavy-hitters
print("co-occurrence matrix built:", V.shape)

**PREDICT before running:** the three nearest neighbors of "tuesday"? of "hotel"? of "expensive"? Commit out loud.

In [ ]:
def neighbors(word, k=5):
    u = V[v2i[word]]
    sims = V @ u / (np.linalg.norm(V, axis=1) * np.linalg.norm(u) + 1e-9)
    order = np.argsort(-sims)
    return [(vocab[i], round(float(sims[i]), 2)) for i in order if vocab[i] != word][:k]

for q in ["tuesday", "hotel", "expensive"]:
    if q in v2i:
        print(f"{q:>10} -> {neighbors(q)}")

Nobody told the matrix that weekdays form a family or that price words travel together — *counting context* discovered it. This is meaning-as-geometry, the bedrock idea under every embedding layer in every LLM.

## See the whole map at once
150 dimensions do not fit on a screen. **PCA** (principal component analysis) finds the 2 directions along which the vectors vary most and projects onto them — a shadow that preserves as much structure as a flat picture can. Six lines of numpy:

In [ ]:
Vc = V - V.mean(0)
U, S, Vt = np.linalg.svd(Vc, full_matrices=False)
P = Vc @ Vt[:2].T
fig, ax = plt.subplots(figsize=(11, 7))
show = vocab[:70]
for w in show:
    x_, y_ = P[v2i[w]]
    ax.annotate(w, (x_, y_), fontsize=9)
ax.scatter(P[[v2i[w] for w in show], 0], P[[v2i[w] for w in show], 1], s=8, alpha=0.4)
ax.set_title("word map from OUR calls (PCA of co-occurrence vectors)")
ax.set_xlabel("principal direction 1"); ax.set_ylabel("principal direction 2"); plt.show()

(How to read: position has no absolute meaning — only *proximity* does. Hunt for: the day-of-week cluster, food/restaurant territory, booking verbs. Words sharing a neighborhood share contexts in real Indian service calls transcribed by a real, garbling ASR.)

## From counted to learned
Real systems do not count — they **learn** the vectors: word2vec (2013) trained small networks to predict context words, and the vectors became famous (king − man + woman ≈ queen). LLMs take the final step: the **token embedding layer** (every token → a learned vector) is literally layer one of the machine you will meet in Part C, learned jointly with everything else by the B1/B2 loop.

## Exercise
Pick three words from `vocab` yourself (print it). For each, predict neighbors, then query. At least one result will be junk — ASR garble or a stopword-ish term. Explain *why* its contexts are uninformative.

In [ ]:
print(vocab)
for q in ["train", "people", "phone"]:        # replace with your three
    if q in v2i:
        print(f"{q:>9} -> {neighbors(q)}")

## Self-check
1. Why are one-hot vectors meaning-blind, in geometric terms?
2. State the distributional hypothesis in one sentence and name the operation that exploits it here.
3. What does cosine similarity measure, and why normalize by the norms?
4. What did PCA buy us, and what did it cost?
5. **Gotcha:** two true synonyms never co-occur *with each other*. Does that break this method?

<details><summary>Answers</summary>

1. All pairs are orthogonal — equal distance everywhere, so the geometry encodes identity only, zero similarity structure.
2. Words in similar contexts mean similar things; we exploit it by counting context windows so rows of the matrix become comparable.
3. The angle between vectors (direction match), ignoring magnitude — frequent words would otherwise dominate raw dot products.
4. A viewable 2D shadow preserving maximal variance; the cost is everything in the discarded 148 directions (some structure is invisible).
5. No — synonyms are similar because they co-occur with the same *third* words (each other's contexts overlap), not because they co-occur together. That is second-order similarity, exactly what the row-vectors capture.
</details>